In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

import os
import getpass

if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass("Enter Groq API Key: ")
if not os.environ.get('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = getpass.getpass("Enter Google API Key: ")
if not os.environ.get('TAVILY_API_KEY'):
    os.environ['TAVILY_API_KEY'] = getpass.getpass("Enter Tavily API Key: ")

Enter Groq API Key:  ········
Enter Google API Key:  ········
Enter Tavily API Key:  ········


In [2]:
# Dense vectorstore + ColBERT backend — the router needs to be able
# to dispatch to EVERY strategy, so every backend has to be loaded
# here, not just one.
import json
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from rag_lab.retrieval.colbert_backend import ColBERTVectorStoreShim

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorstore = Chroma(
    embedding_function=embeddings,
    persist_directory= "~/Codes/rag-lab/data/chroma_naive_gemini",
)
print("Dense chunks in store:", vectorstore._collection.count())

with open(project_root / "data" / "id_to_text.json") as f:
    id_to_text = json.load(f)
colbert_vectorstore = ColBERTVectorStoreShim(
    index_folder=str(project_root / "data" / "colbert_index"),
    id_to_text=id_to_text,
)
print("ColBERT chunks available:", len(id_to_text))

/home/mohnish/my-jupyter-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dense chunks in store: 576


Loading weights: 100%|██████████████████████████████████| 199/199 [00:00<00:00, 533.98it/s]


ColBERT chunks available: 576


In [3]:
# LLMs. Three distinct roles, matching the reasoning already
# established in crag.py: generation can use whatever model, but
# anything needing STRUCTURED OUTPUT (grading, query rewriting, and
# now routing) needs a model with reliable tool-calling support.
# Using the models already proven working across 09_colbert_eval.ipynb
# and 10_crag.ipynb, rather than generation.ipynb's still-unresolved
# Ollama host.
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI

gen_llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)       # final answer generation, all strategies
support_llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash") # grading / rewriting / ROUTING

In [4]:
# Build all 8 strategies. Reuses exactly the classes already built
# across this project — this is literally generation.ipynb's
# strategies dict, extended with colbert and crag.
from rag_lab.strategies.naive import NaiveStrategy
from rag_lab.strategies.multi_query import MultiQueryStrategy
from rag_lab.strategies.rag_fusion import RagFusionStrategy
from rag_lab.strategies.hyde import HydeStrategy
from rag_lab.strategies.step_back import StepBackStrategy
from rag_lab.strategies.decomposition import DecompositionStrategy
from rag_lab.strategies.colbert_strategy import ColBERTStrategy
from rag_lab.strategies.crag import CRAGStrategy

strategies = {
    "naive": NaiveStrategy(vectorstore, llm=gen_llm),
    "multi_query": MultiQueryStrategy(vectorstore, llm=gen_llm),
    "rag_fusion": RagFusionStrategy(vectorstore, llm=gen_llm),
    "hyde": HydeStrategy(vectorstore, llm=gen_llm),
    "step_back": StepBackStrategy(vectorstore, llm=gen_llm),
    "decomposition": DecompositionStrategy(vectorstore, llm=gen_llm),
    "colbert": ColBERTStrategy(colbert_vectorstore, llm=gen_llm),
    "crag": CRAGStrategy(vectorstore, llm=gen_llm, grader_llm=support_llm),
}
print(f"{len(strategies)} strategies ready: {list(strategies)}")

8 strategies ready: ['naive', 'multi_query', 'rag_fusion', 'hyde', 'step_back', 'decomposition', 'colbert', 'crag']


In [5]:
# LogicalRouter — inline prototype (matches this project's established
# pattern: define here for readability, import the committed version
# from rag_lab/strategies/router.py below for the actual run).
#
# NOTE: the `description` text inside RouteQuery is REASONED DEFAULTS,
# not yet validated against this project's actual eval_results.jsonl
# numbers (see the validation cell further down). Tune this prose
# against real per-(strategy, question_type) performance before
# trusting it in anything resembling production.
from typing import Dict, List, Literal, Optional, Protocol
from langchain_core.documents import Document
from langchain_core.language_models import BaseChatModel
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

StrategyName = Literal[
    "naive", "multi_query", "rag_fusion", "hyde",
    "step_back", "decomposition", "colbert", "crag",
]


class RouteQuery(BaseModel):
    """Route a user question to the most suitable RAG strategy for a
    corpus of physics-informed neural network (PINN) and graph neural
    network (GNN) research papers."""

    strategy: StrategyName = Field(
        ...,
        description=(
            "Pick exactly one:\n"
            "- naive: a simple, narrow, single-fact question, directly "
            "answerable from one passage, phrased close to how the "
            "corpus itself would state it.\n"
            "- multi_query: the question is ambiguous, vaguely phrased, "
            "or could reasonably be interpreted a few different ways.\n"
            "- rag_fusion: similar to multi_query, but when you want "
            "the single MOST robust answer via consensus ranking across "
            "phrasings — prefer over multi_query for borderline/noisy questions.\n"
            "- hyde: the question is phrased very differently "
            "(colloquially, abstractly) from how a technical paper "
            "would state the answer.\n"
            "- step_back: the question is VERY narrow/specific (a "
            "single number or parameter from one paper) and would "
            "benefit from broader conceptual context first.\n"
            "- decomposition: the question has multiple distinct parts "
            "or requires comparing/connecting findings ACROSS more than "
            "one paper.\n"
            "- colbert: the question hinges on precise terminology, a "
            "specific named method/author/acronym, or rare technical "
            "terms where exact wording matters.\n"
            "- crag: the question might not be answerable from this "
            "corpus at all (asks about something current/external, a "
            "tool version, or a topic outside PINN/GNN research) and "
            "needs a web-search safety net."
        ),
    )


class LogicalRouter:
    def __init__(self, strategies: Dict[StrategyName, object], router_llm: BaseChatModel):
        expected = set(RouteQuery.model_fields["strategy"].annotation.__args__)
        missing = expected - set(strategies)
        if missing:
            raise ValueError(f"strategies dict is missing required keys: {missing}")
        self.strategies = strategies
        self.structured_router_llm = router_llm.with_structured_output(RouteQuery)
        self.router_prompt = ChatPromptTemplate.from_messages([
            ("system",
             "You are an expert at routing a user question to the most "
             "suitable retrieval-augmented generation strategy for a "
             "corpus of physics-informed neural network (PINN) and "
             "graph neural network (GNN) research papers."),
            ("human", "{question}"),
        ])
        self.router_chain = self.router_prompt | self.structured_router_llm
        self.last_route: Optional[str] = None

    def choose(self, query: str) -> StrategyName:
        result = self.router_chain.invoke({"question": query})
        self.last_route = result.strategy
        return result.strategy

    def retrieve(self, query: str) -> List[Document]:
        chosen = self.choose(query)
        return self.strategies[chosen].retrieve(query)

    def run(self, query: str) -> str:
        chosen = self.choose(query)
        return self.strategies[chosen].run(query)

In [6]:
# Smoke test on four questions pulled from this project's own
# notebooks — deliberately varied, should plausibly route to four
# DIFFERENT strategies if the category descriptions are doing their job.
router = LogicalRouter(strategies=strategies, router_llm=support_llm)

test_questions = [
    "Why do PINNs struggle with irregular geometry?",                                        # ambiguous/broad -> multi_query or rag_fusion?
    "What k-factor did Decke et al. use for their ChebConv layers?",                          # narrow, specific -> step_back or colbert?
    "How does Qian et al.'s CUDA Graph fix relate to the bottleneck MaxK-GNN identifies?",    # cross-paper -> decomposition?
    "What is the current version of PyTorch?",                                                 # out-of-corpus -> crag?
]

for q in test_questions:
    chosen = router.choose(q)
    print(f"[{chosen}]  {q}")

[hyde]  Why do PINNs struggle with irregular geometry?
[step_back]  What k-factor did Decke et al. use for their ChebConv layers?
[decomposition]  How does Qian et al.'s CUDA Graph fix relate to the bottleneck MaxK-GNN identifies?
[crag]  What is the current version of PyTorch?


In [7]:
# Run one end-to-end to confirm the full retrieve->run path works
# through the router, not just the classification step.
print(router.run(test_questions[0]))

PINNs are built on plain fully‑connected (FC) neural networks and on a loss that couples the data‑fit term with the PDE‑residual evaluated at a set of collocation points.  
When the physical domain is irregular (non‑convex, with complicated boundaries, or composed of many sub‑domains) several of the intrinsic limitations of this formulation become acute:

1. **Spectral‑bias of FC nets** – Fully‑connected architectures are known to favour low‑frequency functions and to have difficulty learning high‑frequency or multi‑scale features [29‑31].  In an irregular geometry the solution often exhibits sharp variations near corners, re‑entrant edges or narrow channels, which the network cannot represent efficiently. This leads to poor accuracy and unstable training.

2. **Massive collocation‑point requirement** – To enforce the PDE everywhere, PINNs need a large number of collocation points.  In a complex shape it is hard to place enough points uniformly while respecting the geometry; the number

In [8]:
# Validation bridge — NOT a numeric quality score (scoring.ipynb's
# actual metric isn't available here to reproduce), but a quick way
# to eyeball, per question_type, how each strategy's answers actually
# compare on your real eval set — use this to decide whether the
# RouteQuery descriptions above match reality, and edit them if not.
import json
from collections import defaultdict

with open(project_root / "eval" / "eval_results.jsonl") as f:
    rows = [json.loads(line) for line in f]

by_question = defaultdict(dict)
meta = {}
for r in rows:
    by_question[r["id"]][r["strategy"]] = r["answer"]
    meta[r["id"]] = (r["question"], r["question_type"])

# Pick a couple of cross_document questions to inspect side by side —
# this is exactly the category decomposition/step_back/crag are
# supposed to help with, so a quick read here tells you fast whether
# the router's instinct to prefer decomposition for these actually
# matches which strategy reads best on YOUR corpus.
cross_doc_ids = [qid for qid, (_, qtype) in meta.items() if qtype == "cross_document"][:2]
for qid in cross_doc_ids:
    question, qtype = meta[qid]
    print(f"=== [{qtype}] {question} ===")
    for strategy_name, answer in by_question[qid].items():
        print(f"\n--- {strategy_name} ---")
        print(answer[:300])
    print("\n" + "=" * 80 + "\n")

KeyError: 'answer'

In [ ]:
# Import the committed version (save the class above into
# rag_lab/strategies/router.py first) for the real run.
from rag_lab.strategies.router import LogicalRouter as CommittedLogicalRouter

router = CommittedLogicalRouter(strategies=strategies, router_llm=support_llm)
print(router.run("What is the current version of PyTorch?"))
print("\nRouted to:", router.last_route)